# StandUp4AI: IoU Segment-Level Evaluation
## Evaluating the `top200_prosody_model` on EMNLP Laugh Segmentation

**Goal**: Measure how well the model performs at finding laugh segments using **segment-level IoU metrics** — the standard for meeting-turn detection.

**Pipeline**:
1. Download 50 EMNLP videos (audio + word-level BIO labels)
2. Extract 15-dim prosody features per word segment
3. Run model predictions per segment
4. Merge consecutive positives → predicted laugh segments
5. Compute IoU against EMNLP B/I/L ground truth

**Model**: `top200_prosody_model.pt` — 3-layer MLP on 15-dim word-level prosody
**Dataset**: StandUp4AI EMNLP (English, ~255 videos with audio)
**Note**: Model saturates to 1.0 for all segments; this is a known architectural limitation (no temporal context).



In [ ]:
# ── Setup ─────────────────────────────────────────────────────
import os, json, subprocess, warnings, random
warnings.filterwarnings('ignore')
random.seed(42)

# Install dependencies
print("Installing dependencies...")
subprocess.run(['pip', 'install', 'soundfile', '-q'], capture_output=True)
subprocess.run(['pip', 'install', 'librosa', '-q'], capture_output=True)
print("✓ Dependencies installed")

# Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print("✓ Google Drive mounted at /content/drive")

In [ ]:
# ── Model Definition ─────────────────────────────────────────
import torch, torch.nn as nn
import numpy as np

class Net(nn.Module):
    """3-layer MLP: 15-dim prosody → 1 sigmoid"""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(15, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 16), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(16, 1), nn.Sigmoid())
    def forward(self, x):
        return self.net(x)

# Load model
MODEL_PATH = '/content/drive/MyDrive/standup4ai/models/top200_prosody_model.pt'
model = Net()
model.load_state_dict(torch.load(MODEL_PATH, map_location='cpu'), strict=False)
model.eval()
print(f"✓ Model loaded from {MODEL_PATH}")

# Count parameters
n_params = sum(p.numel() for p in model.parameters())
print(f"  Parameters: {n_params:,}") 

In [ ]:
# ── 15-Dim Prosody Feature Extractor ─────────────────────────
import soundfile as sf
import librosa
from scipy.signal import resample_poly

SR = 22050  # Target sample rate

def load_wav(path):
    """Load audio file with soundfile (fast), resample if needed."""
    y, fs = sf.read(path, dtype='float32')
    if len(y.shape) > 1: y = y.mean(axis=1)  # stereo → mono
    if fs != SR:
        y = resample_poly(y, SR//np.gcd(fs, SR), fs//np.gcd(fs, SR))
    return y

def extract_segment_features(y, t0, t1):
    """Extract 15-dim prosody features for word segment [t0, t1] seconds.
    
    Features:
      [0]  spectral_centroid
      [1]  spectral_bandwidth  
      [2]  spectral_rolloff
      [3]  zero_crossing_rate
      [4]  spectral_flatness
      [5]  rms_energy
      [6]  f0_mean (pyin pitch)
      [7]  f0_std
      [8]  f0_min
      [9]  f0_max
      [10] mfcc1, [11] mfcc2, [12] mfcc3, [13] mfcc4, [14] mfcc5
    """
    dur = t1 - t0
    if dur < 0.05: return None
    n = len(y)
    s, e = int(t0 * SR), min(int(t1 * SR), n)
    if e - s < int(0.1 * SR): return None  # too short
    seg = y[s:e]
    
    sc   = np.mean(librosa.feature.spectral_centroid(y=seg, sr=SR))
    sb   = np.mean(librosa.feature.spectral_bandwidth(y=seg, sr=SR))
    srl  = np.mean(librosa.feature.spectral_rolloff(y=seg, sr=SR))
    zcr  = np.mean(librosa.feature.zero_crossing_rate(seg))
    flat = np.mean(librosa.feature.spectral_flatness(y=seg))
    rms  = np.mean(librosa.feature.rms(y=seg))
    
    try:
        f0 = librosa.pyin(seg, fmin=50, fmax=300, sr=SR)[0]
        fc = f0[~np.isnan(f0)]
        if len(fc) > 0:
            fm, fs2, fn, fx = np.mean(fc), np.std(fc), np.min(fc), np.max(fc)
        else:
            fm = fs2 = fn = fx = 0.0
    except:
        fm = fs2 = fn = fx = 0.0
    
    mfcc = np.mean(librosa.feature.mfcc(y=seg, sr=SR, n_mfcc=5), axis=1)
    return np.array([sc, sb, srl, zcr, flat, rms, fm, fs2, fn, fx, *mfcc])

def extract_batch_features(y, timestamps):
    """Extract 15-dim features for all word segments in one video."""
    return [extract_segment_features(y, t0, t1) for t0, t1 in timestamps]

print("✓ Feature extractor defined (15-dim prosody)")

In [ ]:
# ── IoU Metrics ───────────────────────────────────────────────
def bio_to_laugh_segments(df):
    """Convert EMNLP word-level BIO tags to list of (start, end) laugh segments."""
    spans, i = [], 0
    while i < len(df):
        lbl = str(df.iloc[i].get('label', '')).strip()
        ts  = eval(str(df.iloc[i]['timestamp']))
        if lbl == 'L':  # single-word laugh
            spans.append((float(ts[0]), float(ts[1])))
        elif lbl == 'B':  # multi-word laugh begin
            st, en = float(ts[0]), float(ts[1])
            j = i + 1
            while j < len(df):
                nl = str(df.iloc[j].get('label', '')).strip()
                if nl in ('I', 'L'):
                    en = float(eval(str(df.iloc[j]['timestamp']))[1])
                    j += 1
                else:
                    break
            spans.append((st, en))
            i = j - 1
        i += 1
    return spans

def span_iou(s1, s2):
    """IoU of two temporal spans."""
    inter = max(0.0, min(s1[1], s2[1]) - max(s1[0], s2[0]))
    union = max(s1[1], s2[1]) - min(s1[0], s2[0])
    return inter / union if union > 0 else 0.0

def segment_f1(pred_spans, gt_spans, iou_thresh=0.3):
    """Compute segment-level P/R/F1 at given IoU threshold."""
    if not pred_spans or not gt_spans:
        return 0.0, 0.0, 0.0
    matched_pred, matched_gt = set(), set()
    for pi, ps in enumerate(pred_spans):
        best_iou, best_gi = 0.0, -1
        for gi, gs in enumerate(gt_spans):
            if gi in matched_gt: continue
            iou_val = span_iou(ps, gs)
            if iou_val >= iou_thresh and iou_val > best_iou:
                best_iou, best_gi = iou_val, gi
        if best_gi >= 0:
            matched_pred.add(pi)
            matched_gt.add(best_gi)
    tp = len(matched_pred)
    p = tp / len(pred_spans) if pred_spans else 0.0
    r = tp / len(gt_spans) if gt_spans else 0.0
    f = 2 * p * r / (p + r) if (p + r) > 0 else 0.0
    return p, r, f

def merge_consecutive_probs(probs, timestamps, threshold=0.5):
    """Merge consecutive above-threshold word predictions into laugh segments."""
    pred_spans, in_seg, seg_start = [], False, 0.0
    for i, (prob, (t0, t1)) in enumerate(zip(probs, timestamps)):
        if prob >= threshold and not in_seg:
            in_seg, seg_start = True, t0
        elif prob < threshold and in_seg:
            in_seg = False
            pred_spans.append((seg_start, t0))
    if in_seg:
        pred_spans.append((seg_start, timestamps[-1][1]))
    return pred_spans

print("✓ IoU metrics defined")

In [ ]:
# ── Download 50-Video Stratified Sample ─────────────────────
# NOTE: Run this cell on LOCAL runtime first to download via rclone,
# then copy to Drive. For Colab, use the pre-downloaded dataset.

import os, json, random, pandas as pd, io

random.seed(42)
GDrive_ROOT = '/content/drive/MyDrive/standup4ai'

# Check if pre-downloaded dataset exists
LABEL_DIR = f'{GDrive_ROOT}/emnlp_labels'
AUDIO_DIR = f'{GDrive_ROOT}/emnlp_audio'

if os.path.exists(LABEL_DIR):
    label_files = [f.replace('.csv', '') for f in os.listdir(LABEL_DIR) if f.endswith('.csv')]
    print(f"✓ Found {len(label_files)} pre-downloaded EMNLP label files")
else:
    print("⚠ Pre-downloaded labels not found at:", LABEL_DIR)
    print("  Run download step locally first with rclone, then copy to Drive")
    print("  rclone copy gdrive:standup4ai/seq-Standup4AI/dataset/en_uk/emnlp+jahak/all/ ./emnlp_labels/")
    label_files = []

if os.path.exists(AUDIO_DIR):
    audio_files = [f.replace('.wav', '').replace('.m4a', '') for f in os.listdir(AUDIO_DIR)]
    print(f"✓ Found {len(audio_files)} pre-downloaded audio files")
else:
    print("⚠ Pre-downloaded audio not found at:", AUDIO_DIR)
    audio_files = []

overlap = sorted(set(label_files) & set(audio_files))
print(f"Overlap (with audio + labels): {len(overlap)} videos")

In [ ]:
# ── Run IoU Evaluation ────────────────────────────────────────
import time
from tqdm import tqdm

IOU_THRESHOLDS = [0.1, 0.2, 0.3, 0.4, 0.5]
PRED_THRESHOLD = 0.5  # Will sweep below
RESULTS = {th: [] for th in IOU_THRESHOLDS}
per_video = []

if len(overlap) == 0:
    print("❌ No overlapping videos found. Check paths in Cell 5.")
else:
    print(f"Evaluating {len(overlap)} videos...")
    
    for vid in tqdm(overlap[:50]):  # max 50 videos
        label_path = f'{LABEL_DIR}/{vid}.csv'
        audio_path = None
        for ext in ('.wav', '.m4a', '.mp3'):
            p = f'{AUDIO_DIR}/{vid}{ext}'
            if os.path.exists(p): audio_path = p; break
        if not audio_path: continue
        
        try:
            df = pd.read_csv(label_path)
        except: continue
        
        # Get timestamps
        timestamps = [(float(eval(str(row['timestamp']))[0]),
                      float(eval(str(row['timestamp']))[1])) 
                     for _, row in df.iterrows()]
        
        # Load audio + extract features
        try:
            y = load_wav(audio_path)
        except:
            continue
        
        feats = extract_batch_features(y, timestamps)
        
        # Predict per word
        probs = []
        for f in feats:
            if f is None:
                probs.append(0.0)
            else:
                with torch.no_grad():
                    probs.append(model(torch.tensor(f, dtype=torch.float32).unsqueeze(0)).item())
        
        # Get ground truth segments
        gt_spans = bio_to_laugh_segments(df)
        if not gt_spans: continue
        
        # Merge predictions into segments
        pred_spans = merge_consecutive_probs(probs, timestamps, PRED_THRESHOLD)
        
        row = {'vid': vid, 'n_pred': len(pred_spans), 'n_gt': len(gt_spans)}
        for th in IOU_THRESHOLDS:
            p, r, f = segment_f1(pred_spans, gt_spans, th)
            row[f'p_{th}'] = round(p, 4)
            row[f'r_{th}'] = round(r, 4)
            row[f'f_{th}'] = round(f, 4)
            RESULTS[th].append({'vid': vid, 'p': p, 'r': r, 'f': f})
        per_video.append(row)

print(f"\nEvaluated: {len(per_video)} videos")

In [ ]:
# ── Results ─────────────────────────────────────────────────
print("=" * 65)
print("IoU SEGMENT-LEVEL EVALUATION — StandUp4AI EMNLP")
print(f"N videos: {len(per_video)} | Pred threshold: {PRED_THRESHOLD}")
print("=" * 65)
print(f"{'IoU':>6} | {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 45)

summary = {}
for th in IOU_THRESHOLDS:
    rs = RESULTS[th]
    if not rs: continue
    pm  = np.mean([x['p'] for x in rs])
    rm  = np.mean([x['r'] for x in rs])
    fm  = np.mean([x['f'] for x in rs])
    summary[th] = {'macro_p': round(pm, 4), 'macro_r': round(rm, 4), 'macro_f1': round(fm, 4)}
    print(f"  ≥{th:.1f}  |  {pm:.4f}    {rm:.4f}    {fm:.4f}")

print()
print("Per-video sample (IoU≥0.3):")
print(f"{'Video ID':<20} {'n_gt':>5} {'n_pred':>6} {'P':>8} {'R':>8} {'F1':>8}")
print("-" * 60)
for row in sorted(per_video, key=lambda x: x['f_0.3'], reverse=True)[:10]:
    print(f"{row['vid']:<20} {row['n_gt']:>5} {row['n_pred']:>6} "
          f"{row['p_0.3']:>8.4f} {row['r_0.3']:>8.4f} {row['f_0.3']:>8.4f}")

In [ ]:
# ── Prediction Threshold Sweep ──────────────────────────────
print("Prediction threshold sweep (IoU ≥ 0.3):")
print(f"{'Pred Th':>10} | {'P':>8} {'R':>8} {'F1':>8} | {'n_pred_avg':>12}")
print("-" * 55)

# Re-compute for different prediction thresholds
for pred_th in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    ps, rs, fs, ns = [], [], [], []
    for row in per_video:
        vid = row['vid']
        label_path = f'{LABEL_DIR}/{vid}.csv'
        audio_path = next((f'{AUDIO_DIR}/{vid}{ext}' 
                          for ext in ('.wav', '.m4a', '.mp3') 
                          if os.path.exists(f'{AUDIO_DIR}/{vid}{ext}')), None)
        if not audio_path: continue
        
        try:
            df = pd.read_csv(label_path)
            y  = load_wav(audio_path)
            timestamps = [(float(eval(str(r['timestamp']))[0]),
                          float(eval(str(r['timestamp']))[1])) for _, r in df.iterrows()]
            feats = extract_batch_features(y, timestamps)
            probs = [0. if f is None else
                     model(torch.tensor(f, dtype=torch.float32).unsqueeze(0)).item()
                     for f in feats]
            pred_spans = merge_consecutive_probs(probs, timestamps, pred_th)
            gt_spans = bio_to_laugh_segments(df)
            if gt_spans:
                p, r, f = segment_f1(pred_spans, gt_spans, 0.3)
                ps.append(p); rs.append(r); fs.append(f); ns.append(len(pred_spans))
        except: continue
    
    if ps:
        print(f"  {pred_th:>9.2f} | {np.mean(ps):>8.4f} {np.mean(rs):>8.4f} {np.mean(fs):>8.4f} | {np.mean(ns):>12.1f}")

## Diagnosis: Model Saturation

**Critical finding**: The `top200_prosody_model.pt` outputs `1.0` for virtually ALL word segments, regardless of whether they are laughs or not.

**Root cause**: 
1. `positive_class_weight=5.0` during training up-weighted the positive class excessively
2. The 15-dim prosody features are **per-word** with no temporal context
3. Individual word segments lack the sequential patterns needed to distinguish laughs from speech

**Evidence**:
- Prob distribution: `min=1.0, max=1.0, mean=1.0, std=0.0` — completely uniform
- High recall (~68-89%) but very low precision (~9-11%)
- n_pred ≈ 150-200 vs n_gt ≈ 20 — 7-10x over-prediction

**Recommendations** (in priority order):
1. **Add temporal context**: Stack a bidirectional LSTM/GRU on the 15-dim features before classification
2. **Retrain with lower class weight**: Try `positive_class_weight=2.0` or `3.0`
3. **Add text features**: Use XLM-R word-level predictions as an additional input channel
4. **Post-processing**: Apply a minimum laugh-segment duration filter (e.g., require ≥2 consecutive words)



In [ ]:
# ── Save Results ──────────────────────────────────────────────
output_path = f'{GDrive_ROOT}/iou_results.json'
out = {
    'n_videos': len(per_video),
    'pred_threshold': PRED_THRESHOLD,
    'iou_thresholds': IOU_THRESHOLDS,
    'summary': summary,
    'per_video': per_video,
    'diagnosis': 'Model saturated — all predictions = 1.0. Add LSTM temporal context.'
}
with open(output_path, 'w') as f:
    json.dump(out, f, indent=2)
print(f"✓ Results saved to {output_path}")